# HPSA Mental Health Data Cleaning

This notebook explores and cleans the Health Professional Shortage Area (HPSA) Mental Health dataset from the Health Resources and Services Administration (HRSA).

The goal is to understand the structure of the dataset, identify fields relevant to mental health provider shortages and geographic access, review missing or coded values, and prepare a cleaned dataset for later analysis and visualization.

## Data Sources

- `BCD_HPSA_FCT_DET_MH.csv` — Mental Health HPSA data
- `HPSA_DATAMART_METADATA.XLSX` — HRSA metadata and field definitions

## Initial Workflow

1. Load the dataset and metadata
2. Inspect dimensions and column names
3. Review sample records
4. Check duplicates and missing values
5. Review important fields using the metadata
6. Identify variables relevant to the research questions
7. Clean and standardize selected fields
8. Export a cleaned HPSA dataset

In [1]:
import pandas as pd

hpsa = pd.read_csv("../data/raw/HPSA/BCD_HPSA_FCT_DET_MH.csv")

print("Shape:", hpsa.shape)
hpsa.head()

Shape: (40633, 65)


C:\Users\akila\AppData\Local\Temp\ipykernel_81640\2073839174.py:3: DtypeWarning: Columns (0: BHCMIS Organization Identification Number) have mixed types. Specify dtype option on import or set low_memory=False.
  hpsa = pd.read_csv("../data/raw/HPSA/BCD_HPSA_FCT_DET_MH.csv")


,HPSA Name,HPSA ID,Designation Type,HPSA Discipline Class,HPSA Score,Primary State Abbreviation,HPSA Status,HPSA Designation Date,HPSA Designation Last Update Date,Metropolitan Indicator,...,Rural Status Code,State Abbreviation,State and County Federal Information Processing Standard Code,State FIPS Code,State Name,U.S. - Mexico Border 100 Kilometer Indicator,U.S. - Mexico Border County Indicator,Data Warehouse Record Create Date,Data Warehouse Record Create Date Text,Unnamed: 64
0,Northeast Wyoming MHCA,7569685949,High Needs Geographic HPSA,Mental Health,14,WY,Withdrawn,12/27/2000,07/01/2026,Unknown,...,NaN,WY,56011,56,Wyoming,N,N,08/07/2026,2026/08/07,NaN
1,Northeast Wyoming MHCA,7569685949,High Needs Geographic HPSA,Mental Health,14,WY,Withdrawn,12/27/2000,07/01/2026,Unknown,...,NaN,WY,56005,56,Wyoming,N,N,08/07/2026,2026/08/07,NaN
2,Southwest Wyoming,7568613844,High Needs Geographic HPSA,Mental Health,18,WY,Designated,03/20/2012,09/22/2025,Unknown,...,R,WY,56037,56,Wyoming,N,N,08/07/2026,2026/08/07,NaN
3,Southwest Wyoming,7568613844,High Needs Geographic HPSA,Mental Health,18,WY,Designated,03/20/2012,09/22/2025,Unknown,...,R,WY,56039,56,Wyoming,N,N,08/07/2026,2026/08/07,NaN
4,Southwest Wyoming,7568613844,High Needs Geographic HPSA,Mental Health,18,WY,Designated,03/20/2012,09/22/2025,Unknown,...,R,WY,56035,56,Wyoming,N,N,08/07/2026,2026/08/07,NaN


## Initial Dataset Inspection

The raw HPSA Mental Health dataset contains 40,633 rows and 65 columns.

Initial inspection shows that the dataset includes HPSA designation information, shortage scores, status, geographic characteristics, and other detailed fields. Multiple rows may share the same HPSA ID, so the dataset grain needs to be examined before removing duplicates or selecting variables.

The next step is to inspect the full list of columns and determine which fields describe the HPSA designation itself versus more detailed geographic or facility-level records.

In [2]:
hpsa.columns.tolist()

['HPSA Name',
 'HPSA ID',
 'Designation Type',
 'HPSA Discipline Class',
 'HPSA Score',
 'Primary State Abbreviation',
 'HPSA Status',
 'HPSA Designation Date',
 'HPSA Designation Last Update Date',
 'Metropolitan Indicator',
 'HPSA Geography Identification Number',
 'HPSA Degree of Shortage',
 'Withdrawn Date',
 'HPSA FTE',
 'HPSA Designation Population',
 '% of Population Below 100% Poverty',
 'HPSA Formal Ratio',
 'HPSA Population Type',
 'Rural Status',
 'Longitude',
 'Latitude',
 'BHCMIS Organization Identification Number',
 'Break in Designation',
 'Common County Name',
 'Common Postal Code',
 'Common Region Name',
 'Common State Abbreviation',
 'Common State County FIPS Code',
 'Common State FIPS Code',
 'Common State Name',
 'County Equivalent Name',
 'County or County Equivalent Federal Information Processing Standard Code',
 'Discipline Class Number',
 'HPSA Address',
 'HPSA City',
 'HPSA Component Name',
 'HPSA Component Source Identification Number',
 'HPSA Component State 

In [3]:
for i, col in enumerate(hpsa.columns, start=1):
    print(f"{i}. {col}")

1. HPSA Name
2. HPSA ID
3. Designation Type
4. HPSA Discipline Class
5. HPSA Score
6. Primary State Abbreviation
7. HPSA Status
8. HPSA Designation Date
9. HPSA Designation Last Update Date
10. Metropolitan Indicator
11. HPSA Geography Identification Number
12. HPSA Degree of Shortage
13. Withdrawn Date
14. HPSA FTE
15. HPSA Designation Population
16. % of Population Below 100% Poverty
17. HPSA Formal Ratio
18. HPSA Population Type
19. Rural Status
20. Longitude
21. Latitude
22. BHCMIS Organization Identification Number
23. Break in Designation
24. Common County Name
25. Common Postal Code
26. Common Region Name
27. Common State Abbreviation
28. Common State County FIPS Code
29. Common State FIPS Code
30. Common State Name
31. County Equivalent Name
32. County or County Equivalent Federal Information Processing Standard Code
33. Discipline Class Number
34. HPSA Address
35. HPSA City
36. HPSA Component Name
37. HPSA Component Source Identification Number
38. HPSA Component State Abbrevi

## Metadata Review

Before selecting or removing variables, the HRSA metadata file will be reviewed to understand field definitions and distinguish designation-level information from component-level or administrative fields.

This is especially important because multiple rows can share the same HPSA ID, suggesting that one designation may contain multiple geographic or organizational components.

In [4]:
metadata = pd.ExcelFile("../data/raw/HPSA/HPSA_DATAMART_METADATA.XLSX")

metadata.sheet_names

['DD_HPSA_METADATA_VX', 'Data Dictionary']

In [7]:
data_dictionary = pd.read_excel(
    "../data/raw/HPSA/HPSA_DATAMART_METADATA.XLSX",
    sheet_name="Data Dictionary"
)

print("Shape:", data_dictionary.shape)
data_dictionary.head(10)

Shape: (13, 2)


,Column,Description
0,Health Professional Shortage Areas (HPSA) Data...,This entity represents the data dictionary for...
1,Attribute Name,This attribute represents the specific user fr...
2,Physical Column Name,This attribute represents the metadata databas...
3,User Friendly Names,This attribute represents the concatenated lis...
4,Definition,This attribute represents the basic definition...
5,Data Restriction,This attribute represents the description of t...
6,Organization,This attribute represents the name of the sour...
7,Update Frequency,This attribute represents the description of t...
8,Data Type,This attribute represents the classification o...
9,Business Rule / Format,This attribute represents the specific busines...


In [10]:
hpsa_metadata = pd.read_excel(
    "../data/raw/HPSA/HPSA_DATAMART_METADATA.XLSX",
    sheet_name="DD_HPSA_METADATA_VX"
)

print("Shape:", hpsa_metadata.shape)
hpsa_metadata.head(10)

Shape: (100, 12)


,Attribute Name,Physical Column Name,User Friendly Names,Definition,Data Restriction,Organization,Update Frequency,Data Type,Business Rule / Format,HDW Transformation,Valid Values,Definition Update Date
0,Congressional District Name,CONG_DIST_NM,Congressional District Name; Cong. District; C...,This attribute represents the name of the U.S....,Public,U.S. House of Representatives,As Required,varchar(50),None.,(1) For states with two or more representative...,Expected values are text strings for Congressi...,2016/05/27
1,State FIPS Code,STATE_FIPS_CD,State FIPS Code; State Federal Information Pro...,This attribute represents the State Federal In...,Public,National Institute of Standards and Technology...,As Required,varchar(2),None.,This attribute is a common element which may b...,Expected values are two character numeric text...,2020/08/15
2,State Name,STATE_NM,State Name; Component State; Component State N...,This attribute represents the state or territo...,Public,National Institute of Standards and Technology...,As Required,varchar(50),None.,This attribute is a common element which may b...,Expected values are alphabetic text strings re...,2020/08/15
3,U.S. Senator Name (2),US_SENATE_NM2,U.S. Senator Name (2); Name of U.S. Senator; N...,This attribute represents the name of one of t...,Public,U.S. Senate,Monthly,varchar(100),The District of Columbia and the U.S. territor...,This attribute is a common element which may b...,Expected values are text strings containing th...,2018/03/16
4,Data Warehouse Record Create Date,DW_RECORD_CREATE_DT,Data Warehouse Record Create Date; Data As-of-...,This attribute represents the date on which th...,Public,HRSA Data Warehouse,As Required,smalldatetime,None.,(1) System date on which the data set was refr...,Expected values are date time where the time i...,2022/07/19
5,Site Postal Code,SITE_ZIP_CD,Site Postal Code; Facility ZIP Code; Postal Co...,This attribute represents the U.S. Postal Serv...,Public,"HRSA, Bureau of Primary Health Care (BPHC)",Daily,varchar(10),None.,"Changed, as necessary, as part of the HRSA Dat...",Expected values are 5 or 10 character numeric ...,2020/12/23
6,UDS Number,UDS_NUM,UDS Number; Primary Location UDS Number; UDS #,This attribute represents the unique identific...,Public,"HRSA, Bureau of Primary Health Care (BPHC)",Daily,varchar(20),(1) The HRSA Uniform Data System (UDS) Number ...,This attribute is a common element which may b...,Expected values are six or seven character alp...,2019/12/26
7,Contact Telephone Number,CONTACT_PHONE_NUM,Contact Telephone Number; Contact Phone #; Con...,This attribute represents the telephone number...,Public,HRSA Electronic Handbooks (EHB),Daily,varchar(20),None.,"Changed, as necessary, as part of the HRSA Dat...",Expected values are phone numbers with a forma...,2019/12/26
8,Contact E-mail,CONTACT_EMAIL,Contact E-mail; Contact Email,This attribute represents the email address of...,Public,HRSA Electronic Handbooks (EHB),Daily,varchar(50),None.,None.,Expected values are alphanumeric text strings ...,2016/05/27
9,Site Name,SITE_NM,Site Name; Health Center Name; Satellite Name;...,This attribute represents the name of a health...,Public,"HRSA, Bureau of Primary Health Care (BPHC)",Daily,varchar(100),None.,None.,Expected values are text strings containing th...,2016/05/01


In [11]:
hpsa_metadata.columns.tolist()

['Attribute Name',
 'Physical Column Name',
 'User Friendly Names',
 'Definition',
 'Data Restriction',
 'Organization',
 'Update Frequency',
 'Data Type',
 'Business Rule / Format',
 'HDW Transformation',
 'Valid Values',
 'Definition Update Date']

In [12]:
metadata_lookup = hpsa_metadata[
    ["User Friendly Names", "Definition", "Data Type", "Valid Values"]
].copy()

metadata_lookup.head(20)

,User Friendly Names,Definition,Data Type,Valid Values
0,Congressional District Name; Cong. District; C...,This attribute represents the name of the U.S....,varchar(50),Expected values are text strings for Congressi...
1,State FIPS Code; State Federal Information Pro...,This attribute represents the State Federal In...,varchar(2),Expected values are two character numeric text...
2,State Name; Component State; Component State N...,This attribute represents the state or territo...,varchar(50),Expected values are alphabetic text strings re...
3,U.S. Senator Name (2); Name of U.S. Senator; N...,This attribute represents the name of one of t...,varchar(100),Expected values are text strings containing th...
4,Data Warehouse Record Create Date; Data As-of-...,This attribute represents the date on which th...,smalldatetime,Expected values are date time where the time i...
5,Site Postal Code; Facility ZIP Code; Postal Co...,This attribute represents the U.S. Postal Serv...,varchar(10),Expected values are 5 or 10 character numeric ...
6,UDS Number; Primary Location UDS Number; UDS #,This attribute represents the unique identific...,varchar(20),Expected values are six or seven character alp...
7,Contact Telephone Number; Contact Phone #; Con...,This attribute represents the telephone number...,varchar(20),Expected values are phone numbers with a forma...
8,Contact E-mail; Contact Email,This attribute represents the email address of...,varchar(50),Expected values are alphanumeric text strings ...
9,Site Name; Health Center Name; Satellite Name;...,This attribute represents the name of a health...,varchar(100),Expected values are text strings containing th...


In [13]:
metadata_lookup = hpsa_metadata[
    ["User Friendly Names", "Definition", "Data Type", "Valid Values"]
].copy()

metadata_lookup.head(20)

,User Friendly Names,Definition,Data Type,Valid Values
0,Congressional District Name; Cong. District; C...,This attribute represents the name of the U.S....,varchar(50),Expected values are text strings for Congressi...
1,State FIPS Code; State Federal Information Pro...,This attribute represents the State Federal In...,varchar(2),Expected values are two character numeric text...
2,State Name; Component State; Component State N...,This attribute represents the state or territo...,varchar(50),Expected values are alphabetic text strings re...
3,U.S. Senator Name (2); Name of U.S. Senator; N...,This attribute represents the name of one of t...,varchar(100),Expected values are text strings containing th...
4,Data Warehouse Record Create Date; Data As-of-...,This attribute represents the date on which th...,smalldatetime,Expected values are date time where the time i...
5,Site Postal Code; Facility ZIP Code; Postal Co...,This attribute represents the U.S. Postal Serv...,varchar(10),Expected values are 5 or 10 character numeric ...
6,UDS Number; Primary Location UDS Number; UDS #,This attribute represents the unique identific...,varchar(20),Expected values are six or seven character alp...
7,Contact Telephone Number; Contact Phone #; Con...,This attribute represents the telephone number...,varchar(20),Expected values are phone numbers with a forma...
8,Contact E-mail; Contact Email,This attribute represents the email address of...,varchar(50),Expected values are alphanumeric text strings ...
9,Site Name; Health Center Name; Satellite Name;...,This attribute represents the name of a health...,varchar(100),Expected values are text strings containing th...


In [14]:
hpsa_metadata[
    hpsa_metadata["User Friendly Names"].str.contains(
        "HPSA Score|HPSA ID|HPSA Degree of Shortage",
        case=False,
        na=False
    )
][
    ["Attribute Name", "Physical Column Name", "User Friendly Names", "Definition", "Valid Values"]
]

,Attribute Name,Physical Column Name,User Friendly Names,Definition,Valid Values
25,HPSA Source Identification Number,HPSA_SOURCE_ID,HPSA Source Identification Number; HPSA ID; HP...,This attribute represents the primary identifi...,Expected values are alphanumeric text strings ...
27,HPSA Score,HPSA_SCORE,HPSA Score,This attribute represents the Health Professio...,Expected values are integers 0 to 26. For exam...


In [15]:
important_terms = [
    "HPSA Score",
    "HPSA ID",
    "Degree of Shortage",
    "Estimated Underserved",
    "HPSA Shortage",
    "Population",
    "Poverty",
    "Rural Status",
    "County",
    "FIPS"
]

pattern = "|".join(important_terms)

hpsa_metadata[
    hpsa_metadata["User Friendly Names"].str.contains(
        pattern,
        case=False,
        na=False
    )
][
    ["Attribute Name", "Physical Column Name", "User Friendly Names", "Definition", "Valid Values"]
]

,Attribute Name,Physical Column Name,User Friendly Names,Definition,Valid Values
1,State FIPS Code,STATE_FIPS_CD,State FIPS Code; State Federal Information Pro...,This attribute represents the State Federal In...,Expected values are two character numeric text...
22,Site State County FIPS Code,SITE_STATE_COUNTY_FIPS_CD,Site State County FIPS Code; State County FIPS...,"This attribute represents the five character, ...",Expected values are five digit numeric text st...
25,HPSA Source Identification Number,HPSA_SOURCE_ID,HPSA Source Identification Number; HPSA ID; HP...,This attribute represents the primary identifi...,Expected values are alphanumeric text strings ...
27,HPSA Score,HPSA_SCORE,HPSA Score,This attribute represents the Health Professio...,Expected values are integers 0 to 26. For exam...
29,State and County Federal Information Processin...,STATE_COUNTY_FIPS_CD,State and County Federal Information Processin...,This attribute represents the concatenation (c...,Expected values are numeric text strings stric...
49,Geography Code,Geography_Cd,Geography Code; Census Tract Code; CSD FIPS Co...,This attribute represents the code that unique...,"Expected values are 2, 4, 5, 10, 11, or 12 cha..."
62,County,LIST_BOX_COUNTY_NM_STATE_ABBR,County; County and State; County and State Abb...,This attribute represents the proper county na...,"Expected values are text strings, including pu..."
71,County,LIST_BOX_COUNTY_NM_ONLY,County; County Equivalent; County Name; Full C...,This attribute represents the proper county na...,Expected values are text strings for the prope...


In [16]:
important_terms_2 = [
    "Shortage",
    "Underserved",
    "Population",
    "Poverty",
    "Rural",
    "Metropolitan",
    "Designation Type",
    "Status"
]

pattern_2 = "|".join(important_terms_2)

hpsa_metadata[
    hpsa_metadata["User Friendly Names"].str.contains(
        pattern_2,
        case=False,
        na=False
    )
][
    ["Attribute Name", "Physical Column Name", "User Friendly Names", "Definition", "Valid Values"]
]

,Attribute Name,Physical Column Name,User Friendly Names,Definition,Valid Values
10,Urban Rural Description,URBAN_RURAL,Urban Rural Description; Urban/Rural Setting,This attribute represents the indicator of whe...,Expected values are text strings. The valid va...
11,Site Status Description,SITE_STATUS,Site Status Description; Facility Status; Site...,This attribute represents the indicator of whe...,Expected values are text strings. The valid va...
20,Reporting Status Code,REPORTING_CD,Reporting Status Code; Reporting Code; Reporti...,This attribute represents the reporting code t...,Expected values are text strings. The valid va...
21,Reporting Description,REPORTING_DESC,Reporting Description; Reporting Status,This attribute represents the reporting descri...,Expected values are text strings. The valid va...
26,HPSA Status Code,HPSA_STATUS_CD,HPSA Status Code,This attribute represents the one character id...,Expected values are single character codes. Th...
28,HPSA Status Description,HPSA_STATUS_DESC,HPSA Status Description; HPSA Status; Status,This attribute represents the textual descript...,Expected values are alphabetic text strings. T...
46,HPSA Type Code,HPSA_TYP_CD,HPSA Type Code; HPSA Designation Type Code,This attribute represents the geography or fac...,Expected values are alphabetic text strings of...
47,HPSA Type Description,HPSA_TYP_DESC,HPSA Type Description; Designation Type; HPSA ...,This attribute represents the textual descript...,Expected values are alphabetic text strings. T...
69,In U.S. - Mexico Border 100 Kilometer Area,US_MEXICO_BORDER_100KM_DESC,In U.S. - Mexico Border 100 Kilometer Area; In...,This attribute represents the decoded descript...,Expected values are single character text stri...
93,NHSC Approved Site Status,NHSC_APPROVE_DESC,NHSC Approved Site Status; Is NHSC Approved Site,This attribute represents the indicator of whe...,Expected values are text strings. The valid va...


In [17]:
important_terms_3 = [
    "Shortage",
    "Underserved",
    "Population",
    "Poverty",
    "Rural",
    "Metropolitan"
]

pattern_3 = "|".join(important_terms_3)

mask = (
    hpsa_metadata["Attribute Name"].str.contains(pattern_3, case=False, na=False)
    |
    hpsa_metadata["User Friendly Names"].str.contains(pattern_3, case=False, na=False)
)

hpsa_metadata.loc[
    mask,
    [
        "Attribute Name",
        "Physical Column Name",
        "User Friendly Names",
        "Definition",
        "Valid Values"
    ]
]

,Attribute Name,Physical Column Name,User Friendly Names,Definition,Valid Values
10,Urban Rural Description,URBAN_RURAL,Urban Rural Description; Urban/Rural Setting,This attribute represents the indicator of whe...,Expected values are text strings. The valid va...


In [18]:
candidate_cols = [
    "HPSA Score",
    "HPSA Degree of Shortage",
    "HPSA Designation Population",
    "% of Population Below 100% Poverty",
    "HPSA Formal Ratio",
    "HPSA Population Type",
    "Rural Status",
    "Metropolitan Indicator",
    "Common County Name",
    "HPSA Status",
    "Designation Type"
]

for col in candidate_cols:
    if col in hpsa.columns:
        print(f"\n--- {col} ---")
        print(hpsa[col].value_counts(dropna=False).head(15))


--- HPSA Score ---
HPSA Score
17    6615
16    5601
18    5458
0     3821
15    3641
19    2943
12    2441
14    2431
13    1907
11    1638
20    1269
21     711
10     563
9      439
22     257
Name: count, dtype: int64

--- HPSA Degree of Shortage ---
HPSA Degree of Shortage
Not applicable    32943
NaN                6920
12                  469
6                   194
3                    67
0.2000                4
0.0000                3
0.5000                3
0.3000                3
0.7000                3
0.6000                3
0.4000                2
1.1000                2
1.3000                2
2.8000                2
Name: count, dtype: int64

--- HPSA Designation Population ---
HPSA Designation Population
NaN         1369
0.0         1117
337826.0     269
853967.0     252
952882.0     246
311760.0     194
680837.0     189
248398.0     187
287707.0     183
245573.0     170
355403.0     164
614980.0     161
233571.0     151
321136.0     134
186496.0     134
Name: count, dt

In [19]:
summary = []

for col in candidate_cols:
    if col in hpsa.columns:
        summary.append({
            "Column": col,
            "Data Type": str(hpsa[col].dtype),
            "Non-Null": hpsa[col].notna().sum(),
            "Missing": hpsa[col].isna().sum(),
            "Unique Values": hpsa[col].nunique(dropna=True)
        })

pd.DataFrame(summary)

,Column,Data Type,Non-Null,Missing,Unique Values
0,HPSA Score,int64,40633,0,24
1,HPSA Degree of Shortage,str,33713,6920,26
2,HPSA Designation Population,float64,39264,1369,9111
3,% of Population Below 100% Poverty,float64,29692,10941,424
4,HPSA Formal Ratio,str,22251,18382,1733
5,HPSA Population Type,str,32943,7690,12
6,Rural Status,str,38063,2570,5
7,Metropolitan Indicator,str,40633,0,4
8,Common County Name,str,40633,0,3268
9,HPSA Status,str,40633,0,3


In [20]:
for col in [
    "HPSA Degree of Shortage",
    "HPSA Formal Ratio",
    "HPSA Population Type",
    "Rural Status",
    "Metropolitan Indicator",
    "HPSA Status",
    "Designation Type"
]:
    print(f"\n--- {col} ---")
    print(hpsa[col].value_counts(dropna=False))


--- HPSA Degree of Shortage ---
HPSA Degree of Shortage
Not applicable    32943
NaN                6920
12                  469
6                   194
3                    67
0.2000                4
0.0000                3
0.5000                3
0.3000                3
0.7000                3
0.6000                3
0.4000                2
1.1000                2
1.3000                2
2.8000                2
0.1000                2
1.2000                1
1.9000                1
15.4000               1
1.6000                1
3.1000                1
0.9000                1
0.8000                1
7.6000                1
1.7000                1
6.1000                1
4.9000                1
Name: count, dtype: int64

--- HPSA Formal Ratio ---
HPSA Formal Ratio
NaN         18382
114517:1      269
77988:1       252
32236:1       246
84259:1       194
            ...  
69156:1         1
100063:1        1
45945:1         1
42795:1         1
46917:1         1
Name: count, Length: 1734,

In [21]:
categorical_cols = [
    "HPSA Population Type",
    "Rural Status",
    "Metropolitan Indicator",
    "HPSA Status",
    "Designation Type"
]

for col in categorical_cols:
    print(f"\n--- {col} ---")
    display(
        hpsa[col]
        .value_counts(dropna=False)
        .rename_axis(col)
        .reset_index(name="Count")
    )


--- HPSA Population Type ---


,HPSA Population Type,Count
0,Geographic Population,14097
1,Low Income Population HPSA,13428
2,NaN,7690
3,Medicaid Eligible Population HPSA,4268
4,Homeless Population HPSA,415
5,Other Population HPSA,259
6,Low Income Homeless Population HPSA,211
7,Low Income Migrant Farmworker Population HPSA,165
8,Low Income Homeless Migrant Farmworker Populat...,83
9,Low Income Migrant Seasonal Worker Population ...,13



--- Rural Status ---


,Rural Status,Count
0,Non-Rural,24378
1,Rural,10328
2,Partially Rural,2771
3,NaN,2570
4,Unknown,566
5,Not Applicable,20



--- Metropolitan Indicator ---


,Metropolitan Indicator,Count
0,Unknown,34500
1,Metropolitan,3976
2,Non-Metropolitan,2120
3,Frontier,37



--- HPSA Status ---


,HPSA Status,Count
0,Withdrawn,18613
1,Designated,13836
2,Proposed For Withdrawal,8184



--- Designation Type ---


,Designation Type,Count
0,HPSA Population,18856
1,Geographic HPSA,8073
2,High Needs Geographic HPSA,6014
3,Rural Health Clinic,3193
4,Federally Qualified Health Center,1568
5,"Indian Health Service, Tribal Health, and Urba...",1219
6,Correctional Facility,1026
7,Federally Qualified Health Center Look A Like,298
8,State Mental Hospital,212
9,Other Facility,174


## Filter to Current HPSA Designations

The raw file includes active designations, designations proposed for withdrawal, and historical withdrawn designations.

For the primary analysis, the dataset will be filtered to HPSAs with a current status of `Designated`. Withdrawn and proposed-for-withdrawal records will remain available in the raw source file but will not be included in the main cleaned analysis dataset.

In [22]:
hpsa_current = hpsa[hpsa["HPSA Status"] == "Designated"].copy()

print("Raw shape:", hpsa.shape)
print("Current designated shape:", hpsa_current.shape)

Raw shape: (40633, 65)
Current designated shape: (13836, 65)


In [23]:
print("Rows:", len(hpsa_current))
print("Unique HPSA IDs:", hpsa_current["HPSA ID"].nunique())
print("Duplicate HPSA ID rows:", hpsa_current["HPSA ID"].duplicated().sum())

Rows: 13836
Unique HPSA IDs: 6420
Duplicate HPSA ID rows: 7416


In [24]:
hpsa_current["HPSA ID"].value_counts().head(10)

HPSA ID
7066901765    151
7171440011    129
7442010579    126
7552391298    122
7329101749    114
7362607440    110
7266584666    104
7265464602     99
7365289316     93
7368276284     88
Name: count, dtype: int64

In [25]:
cols_to_check = [
    "HPSA ID",
    "HPSA Name",
    "Designation Type",
    "HPSA Score",
    "Primary State Abbreviation",
    "Common County Name",
    "HPSA Geography Identification Number",
    "HPSA Designation Population",
    "HPSA Population Type",
    "Rural Status",
    "Metropolitan Indicator"
]

hpsa_current[
    hpsa_current["HPSA ID"] == 7066901765
][cols_to_check].head(30)

,HPSA ID,HPSA Name,Designation Type,HPSA Score,Primary State Abbreviation,Common County Name,HPSA Geography Identification Number,HPSA Designation Population,HPSA Population Type,Rural Status,Metropolitan Indicator


In [26]:
hpsa_current[
    hpsa_current["HPSA ID"].astype(str) == "7066901765"
][cols_to_check].head(30)

,HPSA ID,HPSA Name,Designation Type,HPSA Score,Primary State Abbreviation,Common County Name,HPSA Geography Identification Number,HPSA Designation Population,HPSA Population Type,Rural Status,Metropolitan Indicator
34029,7066901765,LI - MSSA 66a-d/Bakersfield,HPSA Population,17,CA,"Kern County, CA",06029003209,233571.0,Low Income Population HPSA,Non-Rural,Unknown
34030,7066901765,LI - MSSA 66a-d/Bakersfield,HPSA Population,17,CA,"Kern County, CA",06029003213,233571.0,Low Income Population HPSA,Non-Rural,Unknown
34031,7066901765,LI - MSSA 66a-d/Bakersfield,HPSA Population,17,CA,"Kern County, CA",06029002402,233571.0,Low Income Population HPSA,Non-Rural,Unknown
34032,7066901765,LI - MSSA 66a-d/Bakersfield,HPSA Population,17,CA,"Kern County, CA",06029001002,233571.0,Low Income Population HPSA,Non-Rural,Unknown
34033,7066901765,LI - MSSA 66a-d/Bakersfield,HPSA Population,17,CA,"Kern County, CA",06029003210,233571.0,Low Income Population HPSA,Non-Rural,Unknown
34034,7066901765,LI - MSSA 66a-d/Bakersfield,HPSA Population,17,CA,"Kern County, CA",06029003214,233571.0,Low Income Population HPSA,Non-Rural,Unknown
34035,7066901765,LI - MSSA 66a-d/Bakersfield,HPSA Population,17,CA,"Kern County, CA",06029003808,233571.0,Low Income Population HPSA,Non-Rural,Unknown
34036,7066901765,LI - MSSA 66a-d/Bakersfield,HPSA Population,17,CA,"Kern County, CA",06029000602,233571.0,Low Income Population HPSA,Non-Rural,Unknown
34037,7066901765,LI - MSSA 66a-d/Bakersfield,HPSA Population,17,CA,"Kern County, CA",06029003218,233571.0,Low Income Population HPSA,Non-Rural,Unknown
34038,7066901765,LI - MSSA 66a-d/Bakersfield,HPSA Population,17,CA,"Kern County, CA",06029003818,233571.0,Low Income Population HPSA,Non-Rural,Unknown


## Dataset Grain

The current-designation dataset contains 13,836 rows representing 6,420 unique HPSA IDs.

Inspection of repeated HPSA IDs shows that designation-level characteristics such as HPSA name, score, designation population, population type, state, and county remain constant across repeated records, while the HPSA Geography Identification Number changes.

This indicates that the dataset is structured at the HPSA geographic component level rather than one row per HPSA designation.

Because the geographic component information is needed for later mapping and integration with other geographic datasets, these repeated HPSA IDs will be preserved rather than deduplicated.

In [27]:
for i, col in enumerate(hpsa_current.columns, start=1):
    print(f"{i}. {col}")

1. HPSA Name
2. HPSA ID
3. Designation Type
4. HPSA Discipline Class
5. HPSA Score
6. Primary State Abbreviation
7. HPSA Status
8. HPSA Designation Date
9. HPSA Designation Last Update Date
10. Metropolitan Indicator
11. HPSA Geography Identification Number
12. HPSA Degree of Shortage
13. Withdrawn Date
14. HPSA FTE
15. HPSA Designation Population
16. % of Population Below 100% Poverty
17. HPSA Formal Ratio
18. HPSA Population Type
19. Rural Status
20. Longitude
21. Latitude
22. BHCMIS Organization Identification Number
23. Break in Designation
24. Common County Name
25. Common Postal Code
26. Common Region Name
27. Common State Abbreviation
28. Common State County FIPS Code
29. Common State FIPS Code
30. Common State Name
31. County Equivalent Name
32. County or County Equivalent Federal Information Processing Standard Code
33. Discipline Class Number
34. HPSA Address
35. HPSA City
36. HPSA Component Name
37. HPSA Component Source Identification Number
38. HPSA Component State Abbrevi

In [28]:
for start in range(0, len(hpsa_current.columns), 15):
    print(f"\n--- Columns {start + 1} to {min(start + 15, len(hpsa_current.columns))} ---")
    for i, col in enumerate(
        hpsa_current.columns[start:start + 15],
        start=start + 1
    ):
        print(f"{i}. {col}")


--- Columns 1 to 15 ---
1. HPSA Name
2. HPSA ID
3. Designation Type
4. HPSA Discipline Class
5. HPSA Score
6. Primary State Abbreviation
7. HPSA Status
8. HPSA Designation Date
9. HPSA Designation Last Update Date
10. Metropolitan Indicator
11. HPSA Geography Identification Number
12. HPSA Degree of Shortage
13. Withdrawn Date
14. HPSA FTE
15. HPSA Designation Population

--- Columns 16 to 30 ---
16. % of Population Below 100% Poverty
17. HPSA Formal Ratio
18. HPSA Population Type
19. Rural Status
20. Longitude
21. Latitude
22. BHCMIS Organization Identification Number
23. Break in Designation
24. Common County Name
25. Common Postal Code
26. Common Region Name
27. Common State Abbreviation
28. Common State County FIPS Code
29. Common State FIPS Code
30. Common State Name

--- Columns 31 to 45 ---
31. County Equivalent Name
32. County or County Equivalent Federal Information Processing Standard Code
33. Discipline Class Number
34. HPSA Address
35. HPSA City
36. HPSA Component Name
37.

In [29]:
hpsa_current["Unnamed: 64"].value_counts(dropna=False)

Unnamed: 64
NaN    13836
Name: count, dtype: int64

In [30]:
hpsa_current = hpsa_current.drop(columns=["Unnamed: 64"])

print("Updated shape:", hpsa_current.shape)

Updated shape: (13836, 64)


In [31]:
for i, col in enumerate(hpsa_current.columns, start=1):
    print(f"{i}. {col}")

1. HPSA Name
2. HPSA ID
3. Designation Type
4. HPSA Discipline Class
5. HPSA Score
6. Primary State Abbreviation
7. HPSA Status
8. HPSA Designation Date
9. HPSA Designation Last Update Date
10. Metropolitan Indicator
11. HPSA Geography Identification Number
12. HPSA Degree of Shortage
13. Withdrawn Date
14. HPSA FTE
15. HPSA Designation Population
16. % of Population Below 100% Poverty
17. HPSA Formal Ratio
18. HPSA Population Type
19. Rural Status
20. Longitude
21. Latitude
22. BHCMIS Organization Identification Number
23. Break in Designation
24. Common County Name
25. Common Postal Code
26. Common Region Name
27. Common State Abbreviation
28. Common State County FIPS Code
29. Common State FIPS Code
30. Common State Name
31. County Equivalent Name
32. County or County Equivalent Federal Information Processing Standard Code
33. Discipline Class Number
34. HPSA Address
35. HPSA City
36. HPSA Component Name
37. HPSA Component Source Identification Number
38. HPSA Component State Abbrevi